In [1]:
# =================================================================
# SOTA ISLES-2022: DERNet Resume Training (Fine-Tuning)
# Corrected & Extended Version
# - Uses a deterministic 70/15/15 train/val/test split (seed=42)
# - Resumes from provided weights
# - Fine-tunes for 150 epochs
# - Saves best validation checkpoint to /kaggle/working
# - After training, loads best checkpoint and evaluates on test set
# - Reports validation and test Dice (F1) scores
# =================================================================

!pip install -q monai nibabel scikit-learn einops

import os
import sys
import torch
import warnings
import numpy as np
import nibabel as nib
import nibabel.processing
from collections import defaultdict
from sklearn.model_selection import train_test_split

import torch.nn as nn
import torch.optim as optim
from torch.amp import GradScaler, autocast
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

from monai.losses import DiceFocalLoss
from monai.metrics import DiceMetric
from monai.transforms import (
    Compose, NormalizeIntensityd, RandSpatialCropd,
    RandFlipd, RandRotate90d, CastToTyped, EnsureTyped, SpatialPadd
)
from monai.inferers import sliding_window_inference
from monai.data import decollate_batch

warnings.filterwarnings("ignore")

# --- 1. KAGGLE PATHS & CONFIGURATION ---
CONFIG = {
    "SEARCH_ROOT": "/kaggle/input/datasets/prosenjitmondol/a-stroke-lesion-segmentation-dataset/ISLES-2022",
    "SAVE_DIR": "/kaggle/working/",
    # Path to uploaded Fold 1 weights (resume). Verify this path in Kaggle sidebar.
    "RESUME_WEIGHTS": "/kaggle/input/datasets/prosenjitmondol/fold-01/DERNet_Fold_1.pth",
    "roi_size": (64, 64, 64),
    "batch_size": 1,
    "epochs": 150,       # Updated to 150 as requested
    "lr": 1e-4,
    "device": torch.device("cuda" if torch.cuda.is_available() else "cpu"),
    "seed": 42,
    "patience": 50,      # keep the increased patience
    # Train/Val/Test split ratios (must sum to 1.0)
    "split": {"train": 0.70, "val": 0.15, "test": 0.15},
}

os.makedirs(CONFIG["SAVE_DIR"], exist_ok=True)
print(f"🚀 Initializing DERNet Resume Engine | Fine-tuning to {CONFIG['epochs']} epochs on device {CONFIG['device']}")

# --- 2. DATA PROCESSING ---
def prepare_isles_data(root):
    subjects = defaultdict(dict)
    for dirpath, _, filenames in os.walk(root):
        for f in filenames:
            if f.endswith(('.nii', '.nii.gz')):
                full_path = os.path.join(dirpath, f)
                # try to find a subject id in the path, fallback to directory name
                sub_id = next((p for p in full_path.split(os.sep) if 'sub-' in p.lower()), os.path.basename(dirpath))
                f_l = f.lower()
                if 'dwi' in f_l: subjects[sub_id]['dwi'] = full_path
                elif 'adc' in f_l: subjects[sub_id]['adc'] = full_path
                elif 'flair' in f_l: subjects[sub_id]['flair'] = full_path
                elif any(x in f_l for x in ['msk', 'mask', 'lesion']): subjects[sub_id]['msk'] = full_path
    # return only complete subjects
    data = [f for s, f in subjects.items() if all(k in f for k in ['dwi', 'adc', 'flair', 'msk'])]
    data = sorted(data, key=lambda x: list(x.values())[0])  # deterministic ordering
    return data

class ISLESDataset(Dataset):
    def __init__(self, data, transform=None):
        self.data, self.transform = data, transform
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        p = self.data[idx]
        dwi = nib.load(p['dwi'])
        adc_r = nib.processing.resample_from_to(nib.load(p['adc']), dwi, order=1)
        flr_r = nib.processing.resample_from_to(nib.load(p['flair']), dwi, order=1)
        msk_r = nib.processing.resample_from_to(nib.load(p['msk']), dwi, order=0)

        img = np.stack([np.nan_to_num(dwi.get_fdata()), np.nan_to_num(adc_r.get_fdata()), np.nan_to_num(flr_r.get_fdata())], 0)
        lbl = np.expand_dims(np.nan_to_num(msk_r.get_fdata()), 0)

        del dwi, adc_r, flr_r, msk_r
        d = {"image": img.astype(np.float32), "label": lbl.astype(np.float32)}
        return self.transform(d) if self.transform else d

# --- 3. AUGMENTATIONS ---
xforms = Compose([
    NormalizeIntensityd(keys="image", nonzero=True, channel_wise=True),
    SpatialPadd(keys=["image", "label"], spatial_size=CONFIG["roi_size"]),
    RandSpatialCropd(keys=["image", "label"], roi_size=CONFIG["roi_size"], random_size=False),
    RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=[0, 1, 2]),
    RandRotate90d(keys=["image", "label"], prob=0.5, max_k=3),
    CastToTyped(keys=["image", "label"], dtype=[torch.float32, torch.float32]),
    EnsureTyped(keys=["image", "label"]),
])

# --- 4. DERNet ARCHITECTURE ---
class LSCBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.c3 = nn.Conv3d(in_c, out_c//3, 3, padding=1)
        self.c5 = nn.Conv3d(in_c, out_c//3, 5, padding=2)
        self.c7 = nn.Conv3d(in_c, out_c - 2*(out_c//3), 7, padding=3)
        self.bn, self.ac = nn.InstanceNorm3d(out_c), nn.GELU()
    def forward(self, x):
        return self.ac(self.bn(torch.cat([self.c3(x), self.c5(x), self.c7(x)], 1)))

class BiMambaSim(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.gru = nn.GRU(c, max(1, c//2), batch_first=True, bidirectional=True)
        self.nm = nn.LayerNorm(c)
    def forward(self, x):
        B, C, D, H, W = x.shape
        s, _ = self.gru(x.view(B, C, -1).permute(0, 2, 1))
        return self.nm(s).permute(0, 2, 1).view(B, C, D, H, W) + x

class BAGF(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.sg = nn.Conv3d(c, 1, 1)
        self.cg = nn.Sequential(nn.AdaptiveAvgPool3d(1), nn.Conv3d(c, c, 1), nn.Sigmoid())
        self.fs = nn.Conv3d(c*2, c, 1)
    def forward(self, e, d):
        return self.fs(torch.cat([e * torch.sigmoid(self.sg(e)), d * self.cg(d)], 1))

class DERNet(nn.Module):
    def __init__(self, in_c=3, out_c=1, f=(32, 64, 128)):
        super().__init__()
        self.e1, self.e2, self.e3 = LSCBlock(in_c, f[0]), LSCBlock(f[0], f[1]), LSCBlock(f[1], f[2])
        self.dn, self.bt = nn.MaxPool3d(2), BiMambaSim(f[2])
        self.u2, self.u1 = nn.ConvTranspose3d(f[2], f[1], 2, 2), nn.ConvTranspose3d(f[1], f[0], 2, 2)
        self.f2, self.d2 = BAGF(f[1]), LSCBlock(f[1], f[1])
        self.f1, self.d1 = BAGF(f[0]), LSCBlock(f[0], f[0])
        self.fn = nn.Conv3d(f[0], out_c, 1)
    def forward(self, x):
        x1 = self.e1(x); x2 = self.e2(self.dn(x1)); x3 = self.e3(self.dn(x2))
        b = self.bt(x3)
        y2 = self.d2(self.f2(x2, self.u2(b)))
        y1 = self.d1(self.f1(x1, self.u1(y2)))
        return self.fn(y1)

# --- 5. RESUME WEIGHTS LOGIC ---
def get_resumed_model():
    m = DERNet().to(CONFIG["device"])
    weight_path = CONFIG["RESUME_WEIGHTS"]

    if os.path.exists(weight_path):
        print(f"📥 Loading previous weights from: {weight_path}")
        state = torch.load(weight_path, map_location=CONFIG["device"])
        # allow loading state dicts saved with or without 'module.' prefix
        try:
            m.load_state_dict(state)
        except RuntimeError:
            # try stripping 'module.' if present
            new_state = {}
            for k, v in state.items():
                new_key = k.replace("module.", "") if k.startswith("module.") else k
                new_state[new_key] = v
            m.load_state_dict(new_state)
        print("✅ Weights successfully loaded!")
    else:
        print(f"❌ ERROR: Could not find weights at {weight_path}")
        print("Please check the Kaggle Input path and update CONFIG['RESUME_WEIGHTS'].")
        sys.exit(1)  # Stops the script so it doesn't train from scratch

    return m

# --- 6. TRAIN / VAL / TEST SPLIT (70/15/15) ---
def split_data(data, seed=42):
    # first split train vs temp (train = 70%, temp = 30%)
    train_ratio = CONFIG["split"]["train"]
    val_ratio = CONFIG["split"]["val"]
    test_ratio = CONFIG["split"]["test"]
    assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-6, "Splits must sum to 1.0"

    train_data, temp_data = train_test_split(data, train_size=train_ratio, random_state=seed, shuffle=True)
    # split temp into val and test proportionally
    temp_size = len(temp_data)
    if temp_size == 0:
        return train_data, [], []
    val_size = int(round(val_ratio / (val_ratio + test_ratio) * temp_size))
    val_data = temp_data[:val_size]
    test_data = temp_data[val_size:]
    return train_data, val_data, test_data

# --- 7. FINE-TUNING ENGINE ---
def run():
    torch.manual_seed(CONFIG["seed"])
    np.random.seed(CONFIG["seed"])
    data = prepare_isles_data(CONFIG["SEARCH_ROOT"])
    if len(data) == 0:
        print("❌ No complete subjects found in SEARCH_ROOT. Check dataset path and file naming.")
        return

    train_data, val_data, test_data = split_data(data, seed=CONFIG["seed"])
    print(f"Dataset sizes -> Total: {len(data)} | Train: {len(train_data)} | Val: {len(val_data)} | Test: {len(test_data)}")

    # DataLoaders
    t_ldr = DataLoader(ISLESDataset(train_data, xforms), batch_size=CONFIG["batch_size"], shuffle=True, num_workers=0)
    v_ldr = DataLoader(ISLESDataset(val_data, xforms), batch_size=CONFIG["batch_size"], shuffle=False, num_workers=0)
    test_ldr = DataLoader(ISLESDataset(test_data, xforms), batch_size=CONFIG["batch_size"], shuffle=False, num_workers=0)

    # Loss, metric, model, optimizer, scheduler
    loss_fn = DiceFocalLoss(include_background=False, sigmoid=True, squared_pred=True, gamma=2.0)
    metric = DiceMetric(include_background=False, reduction="mean")
    m = get_resumed_model()
    opt = optim.AdamW(m.parameters(), lr=CONFIG["lr"], weight_decay=1e-4)
    sch = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CONFIG["epochs"])
    scaler = GradScaler()  # correct usage

    best_val = 0.0
    patience_cnt = 0
    best_model_path = os.path.join(CONFIG["SAVE_DIR"], "DERNet_best_val.pth")

    for ep in range(CONFIG["epochs"]):
        print(f"\nEpoch {ep+1:03d}/{CONFIG['epochs']}")
        m.train()
        l_sum = 0.0
        train_steps = 0
        for b in tqdm(t_ldr, desc="Train", leave=False):
            img, msk = b["image"].to(CONFIG["device"]), b["label"].to(CONFIG["device"])
            opt.zero_grad()
            with autocast(enabled=(CONFIG["device"].type == "cuda")):
                out = m(img)
                loss = loss_fn(out, msk)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            l_sum += loss.item()
            train_steps += 1

        if train_steps == 0:
            avg_loss = 0.0
        else:
            avg_loss = l_sum / train_steps

        sch.step()

        # Validation
        m.eval()
        metric.reset()
        with torch.no_grad():
            for vb in tqdm(v_ldr, desc="Val", leave=False):
                vi, vm = vb["image"].to(CONFIG["device"]), vb["label"].to(CONFIG["device"])
                vo = sliding_window_inference(vi, CONFIG["roi_size"], sw_batch_size=4, predictor=m, overlap=0.6)
                # thresholded predictions for metric
                preds = [torch.sigmoid(i) > 0.5 for i in decollate_batch(vo)]
                metric(y_pred=preds, y=vm)
                del vi, vm, vo

        cur_val = metric.aggregate().item() if len(val_data) > 0 else 0.0
        metric.reset()
        print(f"Loss: {avg_loss:.4f} | Val Dice: {cur_val:.4f}")

        # Save best validation model
        if cur_val > best_val:
            best_val = cur_val
            patience_cnt = 0
            torch.save(m.state_dict(), best_model_path)
            print(f"🌟 New best validation Dice: {best_val:.4f} -> saved to {best_model_path}")
        else:
            patience_cnt += 1
            print(f"No improvement. Patience: {patience_cnt}/{CONFIG['patience']}")
            if patience_cnt >= CONFIG["patience"]:
                print(f"🛑 Early stopping triggered after {patience_cnt} epochs without improvement.")
                break

        # free GPU memory
        if CONFIG["device"].type == "cuda":
            torch.cuda.empty_cache()

    # After training: evaluate best model on validation and test sets
    if os.path.exists(best_model_path):
        print(f"\n🔁 Loading best model from {best_model_path} for final evaluation.")
        best_state = torch.load(best_model_path, map_location=CONFIG["device"])
        try:
            m.load_state_dict(best_state)
        except RuntimeError:
            # handle possible 'module.' prefix
            new_state = {}
            for k, v in best_state.items():
                new_key = k.replace("module.", "") if k.startswith("module.") else k
                new_state[new_key] = v
            m.load_state_dict(new_state)
    else:
        print("⚠️ Best model not found; using current model weights for final evaluation.")

    # Evaluate on validation set (report again)
    if len(val_data) > 0:
        m.eval()
        metric.reset()
        with torch.no_grad():
            for vb in tqdm(v_ldr, desc="Final Val Eval", leave=False):
                vi, vm = vb["image"].to(CONFIG["device"]), vb["label"].to(CONFIG["device"])
                vo = sliding_window_inference(vi, CONFIG["roi_size"], sw_batch_size=4, predictor=m, overlap=0.6)
                preds = [torch.sigmoid(i) > 0.5 for i in decollate_batch(vo)]
                metric(y_pred=preds, y=vm)
                del vi, vm, vo
        final_val_dice = metric.aggregate().item()
        metric.reset()
        print(f"\n✅ Final Validation Dice (F1): {final_val_dice:.4f}")
    else:
        final_val_dice = None
        print("\n⚠️ No validation data to evaluate.")

    # Evaluate on test set
    if len(test_data) > 0:
        m.eval()
        metric.reset()
        with torch.no_grad():
            for tb in tqdm(test_ldr, desc="Test Eval", leave=False):
                ti, tm = tb["image"].to(CONFIG["device"]), tb["label"].to(CONFIG["device"])
                to = sliding_window_inference(ti, CONFIG["roi_size"], sw_batch_size=4, predictor=m, overlap=0.6)
                preds = [torch.sigmoid(i) > 0.5 for i in decollate_batch(to)]
                metric(y_pred=preds, y=tm)
                del ti, tm, to
        test_dice = metric.aggregate().item()
        metric.reset()
        print(f"\n🎯 Test Dice (F1): {test_dice:.4f}")
    else:
        test_dice = None
        print("\n⚠️ No test data to evaluate.")

    print("\n--- Summary ---")
    print(f"Best validation Dice saved at: {best_model_path}")
    if final_val_dice is not None:
        print(f"Final Validation Dice (F1): {final_val_dice:.4f}")
    if test_dice is not None:
        print(f"Test Dice (F1): {test_dice:.4f}")

if __name__ == "__main__":
    run()


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 36.5 MB/s eta 0:00:0000:0100:01


<frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
2026-03-08 15:14:15.894345: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772982856.072542      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772982856.130499      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772982856.539767      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772982856.539820      55 computation_placer.cc:1

🚀 Initializing DERNet Resume Engine | Pushing to 200 Epochs

🔥 RESUMING FOLD 1 FINE-TUNING 🔥
📥 Loading previous weights from: /kaggle/input/datasets/prosenjitmondol/fold-01/DERNet_Fold_1.pth
✅ Weights successfully loaded!
Epoch 001/200


Train:   0%|          | 0/200 [00:00<?, ?it/s]

Loss: 0.3175 | Dice: 0.7089 | F1: 0.7089
🌟 New SOTA Checkpoint: 0.7089 🌟
Epoch 002/200


...

Loss: 0.2776 | Dice: 0.7613 | F1: 0.7613
Epoch 056/200


Train:   0%|          | 0/200 [00:00<?, ?it/s]

Loss: 0.2770 | Dice: 0.7497 | F1: 0.7497
Epoch 057/200


Train:   0%|          | 0/200 [00:00<?, ?it/s]

Loss: 0.2618 | Dice: 0.7419 | F1: 0.7419
🛑 Early Stopping reached after 50 epochs without improvement.
